# 01 — Explore snapshot + Tier A features

Follows `feature_engineering_plan.md`.

**This notebook (step 1):** load latest JSON, clean/sort, verify cumulative OI, scaffold Tier A columns.

Do not jump to modeling until Tier A flags align with known phases in `data/predictions/latest.md`.

In [ ]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("../..").resolve()
SNAPSHOT_DIR = ROOT / "data" / "snapshots"
PRED_DIR = ROOT / "data" / "predictions"

print("ROOT:", ROOT)
print("snapshots:", sorted(SNAPSHOT_DIR.glob("*.json")))

## 1. Load latest snapshot

In [ ]:
def load_snapshot(path: Path) -> tuple[dict, pd.DataFrame]:
    meta = json.loads(path.read_text())
    df = pd.DataFrame(meta["rows"])
    return meta, df


latest = max(SNAPSHOT_DIR.glob("*.json"), key=lambda p: p.stat().st_mtime)
meta, raw = load_snapshot(latest)
print("file:", latest.name)
print("scrapedAt:", meta.get("scrapedAt"), "rowCount:", meta.get("rowCount"))
raw.head()

## 2. Clean + chronological sort

Source rows are newest-first. Analysis needs morning → afternoon.

In [ ]:
def parse_time_to_seconds(t) -> int | None:
    if t is None or (isinstance(t, float) and np.isnan(t)):
        return None
    s = str(t).strip().upper()
    if s == "EOD":
        return 16 * 3600  # after last intraday bar
    m = re.match(r"^(\d{1,2}):(\d{2})(?::(\d{2}))?$", s)
    if not m:
        return None
    h, mi, sec = int(m.group(1)), int(m.group(2)), int(m.group(3) or 0)
    return h * 3600 + mi * 60 + sec


def clean_rows(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["_tsec"] = out["time"].map(parse_time_to_seconds)
    out = out.dropna(subset=["_tsec"]).sort_values("_tsec").reset_index(drop=True)
    out["is_eod"] = out["time"].astype(str).str.upper().eq("EOD")
    return out


df = clean_rows(raw)
intra = df.loc[~df["is_eod"]].copy()
print("sorted times:", intra["time"].iloc[0], "→", intra["time"].iloc[-1], "n=", len(intra))
intra[["time", "ltp", "netPCR", "strength", "diffInOI", "chngInCallOI", "chngInPutOI"]].head(8)

## 3. Sanity check: are OI fields cumulative?

If call/put OI mostly increase in absolute level through the day (with occasional declines), treat as cumulative and engineer `d_*` deltas.

In [ ]:
check = intra[["time", "chngInCallOI", "chngInPutOI", "diffInOI"]].copy()
check["call_mono_up"] = check["chngInCallOI"].diff() >= 0
check["put_mono_up"] = check["chngInPutOI"].diff() >= 0
print(
    "fraction of bars where call OI level rose:",
    check["call_mono_up"].mean().round(3),
)
print(
    "fraction of bars where put OI level rose:",
    check["put_mono_up"].mean().round(3),
)

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(intra["time"], intra["chngInCallOI"] / 1e6, label="call OI (M)")
ax.plot(intra["time"], intra["chngInPutOI"] / 1e6, label="put OI (M)")
ax.plot(intra["time"], intra["diffInOI"] / 1e6, label="diff OI (M)", alpha=0.8)
ax.set_title("OI levels through session (expect cumulative shape)")
ax.legend()
ax.tick_params(axis="x", labelrotation=90, labelsize=7)
plt.tight_layout()
plt.show()

## 4. Tier A scaffold (implement next)

Columns to add per plan:

- **A1** `d_call_oi`, `d_put_oi`, `d_diff_oi`, `d_pcr`, `d_strength`, `d_ltp`
- **A2** `pcr_regime`, `strength_regime`, `oi_side`, cross/flip flags
- **A3** `activity_label`
- **A4** `poi_relation`
- **A5** `hl_event`, `hl_level`

Fill these in step-by-step in follow-up cells / `src/features_tier_a.py`.

In [ ]:
def add_velocity_features(frame: pd.DataFrame) -> pd.DataFrame:
    """Tier A1 — bar-over-bar deltas from cumulative series."""
    out = frame.copy()
    out["d_call_oi"] = out["chngInCallOI"].diff()
    out["d_put_oi"] = out["chngInPutOI"].diff()
    out["d_diff_oi"] = out["diffInOI"].diff()
    out["d_pcr"] = out["netPCR"].diff()
    out["d_strength"] = out["strength"].diff()
    out["d_ltp"] = out["ltp"].diff()
    out["d_ltp_pct"] = out["ltp"].pct_change() * 100
    return out


def parse_hl_break(val):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return "none", np.nan
    s = str(val).strip()
    m = re.match(r"^(D\.?H\.?B\.?|D\.?L\.?B\.?)\s*\(([-\d.]+)\)", s, re.I)
    if not m:
        return "none", np.nan
    kind = m.group(1).upper().replace(".", "")
    event = "DHB" if "H" in kind else "DLB"
    return event, float(m.group(2))


feat = add_velocity_features(intra)
hl = feat["dayHLBreak"].map(parse_hl_break)
feat["hl_event"] = [x[0] for x in hl]
feat["hl_level"] = [x[1] for x in hl]

feat[
    [
        "time",
        "ltp",
        "d_ltp",
        "netPCR",
        "d_pcr",
        "strength",
        "d_strength",
        "d_call_oi",
        "d_put_oi",
        "hl_event",
    ]
].head(12)

## 5. Next steps

1. Implement A2–A4 regime / activity / price–OI helpers.
2. Plot `ltp` with vertical lines on `pcr_cross_1`, `strength_flip`, `hl_event`.
3. Compare candidate boundaries to phases in `../data/predictions/latest.md`.
4. Promote stable functions into `src/features_tier_a.py`.